# NTP ICE Skin Sensitization — ML Model Pipeline

**목표**: 피부 감작성 물질 예측 이진 분류 모델 개발

- **입력**: `skin_sensitization_processed.csv` (data_pipeline 출력)
- **디스크립터**: MACCS only (167 bit)
- **모델**: SVM (RBF kernel, class_weight=balanced)
- **출력**: `skin_sensitization_model.joblib`

## 1. 라이브러리 불러오기

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from rdkit import Chem, RDLogger
from rdkit.Chem import MACCSkeys
from rdkit.Chem.SaltRemover import SaltRemover
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                              roc_auc_score, roc_curve,
                              matthews_corrcoef, f1_score)
import joblib

print('라이브러리 로드 완료')

## 2. 전처리된 데이터 로드

In [ ]:
df = pd.read_csv('skin_sensitization_processed.csv')
print(f'데이터 로드: {df.shape}')

maccs_cols   = [c for c in df.columns if c.startswith('MACCS_')]
X_maccs      = df[maccs_cols].values
y            = df['label'].values
idx_trainval = df[df['split'] == 0].index.tolist()
idx_test     = df[df['split'] == 1].index.tolist()

print(f'MACCS shape : {X_maccs.shape}')
print(f'Train+Val   : {len(idx_trainval)}개  |  Test: {len(idx_test)}개')
print(f'Active(1)   : {y.sum()}  Inactive(0): {(y==0).sum()}')

## 3. 분류 모델 학습 — MACCS only + SVM

> 전체 디스크립터·모델 비교는 별도 실험 파일 참조. MACCS + SVM 선택.

In [ ]:
cv_clf   = StratifiedKFold(n_splits=5, shuffle=True)
X_clf_tv = X_maccs[idx_trainval]
y_tv     = y[idx_trainval]

clf_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True))
])

scores = cross_val_score(clf_model, X_clf_tv, y_tv,
                         cv=cv_clf, scoring='balanced_accuracy', n_jobs=-1)
print(f'[MACCS only + SVM] 5-Fold CV Balanced Accuracy')
print(f'  Mean : {scores.mean():.4f}')
print(f'  Std  : {scores.std():.4f}')

### 3-1. Learning Curve

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    Pipeline([('scaler', StandardScaler()),
              ('clf', SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True))]),
    X_clf_tv, y_tv,
    cv=StratifiedKFold(n_splits=5, shuffle=True),
    scoring='balanced_accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

tm = train_scores.mean(axis=1); ts = train_scores.std(axis=1)
vm = val_scores.mean(axis=1);   vs = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(train_sizes, tm, 'o-', color='steelblue', label='Train')
ax.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.15, color='steelblue')
ax.plot(train_sizes, vm, 's-', color='tomato', label='Validation (CV)')
ax.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.15, color='tomato')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('Balanced Accuracy')
ax.set_title('Learning Curve — MACCS only + SVM (Skin Sensitization)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 📊 Learning Curve 결과 분석

학습 곡선은 **과적합(overfitting)** 또는 **데이터 부족** 여부를 진단하는 데 사용된다.

- **Train 점수(파란색)**: 학습 데이터 크기가 증가할수록 다소 감소하는 경향이 있으며, 이는 더 많은 데이터를 학습하면서 모델이 과도하게 맞추지 않게 되기 때문이다.
- **Validation 점수(빨간색)**: 데이터 양이 늘수록 일반화 성능이 향상되며, 두 곡선이 수렴할수록 과적합이 아닌 적절한 모델 복잡도를 의미한다.
- **두 곡선 간 간격이 좁고 수렴**: 모델이 안정적으로 학습되었음을 의미한다.
- **간격이 넓게 유지**: Train >> Validation이면 과적합, 둘 다 낮으면 과소적합(under-fitting) 신호로 더 많은 데이터 또는 더 복잡한 모델이 필요하다.

## 4. 최고 성능 모델 — 테스트셋 평가

In [ ]:
final_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True))
])
final_clf.fit(X_maccs[idx_trainval], y[idx_trainval])

X_test  = X_maccs[idx_test]
y_test  = y[idx_test]
y_proba = final_clf.predict_proba(X_test)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
bal_acc = balanced_accuracy_score(y_test, y_pred)
mcc     = matthews_corrcoef(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
cm      = confusion_matrix(y_test, y_pred)

print(f'테스트셋 성능 [MACCS only + SVM]')
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  ROC-AUC           : {roc_auc:.4f}')
print(f'  MCC               : {mcc:.4f}')
print(f'  F1                : {f1:.4f}')

### 4-1. 결정 임계값 최적화

In [ ]:
thresholds = np.linspace(0.01, 0.99, 199)
bal_accs, sens_list, spec_list = [], [], []

for thr in thresholds:
    y_pred_thr = (y_proba >= thr).astype(int)
    bal_accs.append(balanced_accuracy_score(y_test, y_pred_thr))
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thr).ravel()
    sens_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

best_idx  = int(np.argmax(bal_accs))
best_thr  = thresholds[best_idx]
best_bal  = bal_accs[best_idx]
best_sens = sens_list[best_idx]
best_spec = spec_list[best_idx]

y_pred_opt = (y_proba >= best_thr).astype(int)
cm_opt     = confusion_matrix(y_test, y_pred_opt)

print(f'최적 임계값: {best_thr:.3f}  →  Balanced Accuracy: {best_bal:.4f}')
print(f'  Sensitivity: {best_sens:.4f}  Specificity: {best_spec:.4f}')

### 4-2. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fig. 4  Skin Sensitization 분류 모델 테스트 평가', fontsize=13, fontweight='bold')

sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Inactive(0)', 'Active(1)'],
            yticklabels=['Inactive(0)', 'Active(1)'],
            linewidths=0.5, linecolor='white', cbar=False,
            annot_kws={'size': 14})
axes[0].set_xlabel('예측값'); axes[0].set_ylabel('실제값')
axes[0].set_title(f'Confusion Matrix (thr={best_thr:.2f})')

axes[1].plot(fpr, tpr, color='tomato', lw=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

metrics = {'BalAcc': best_bal, 'ROC-AUC': roc_auc, 'MCC': mcc, 'F1': f1}
colors  = ['steelblue','tomato','seagreen','darkorange']
axes[2].bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', width=0.5)
for i, (k, v) in enumerate(metrics.items()):
    axes[2].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=11)
axes[2].set_ylim(0, 1.15)
axes[2].set_title('테스트셋 성능 지표 - MACCS only + SVM')
axes[2].axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.5)

plt.tight_layout(); plt.show()

### 📊 Fig. 4 결과 분석

**Confusion Matrix(좌)**: 최적 임계값 적용 후 예측 결과를 행(실제)×열(예측) 형태로 나타낸다. 대각선 값(TP, TN)이 클수록 모델 성능이 좋다. 독성학적 관점에서 **FN(실제 감작성 → 비감작성으로 잘못 예측)** 은 안전 위험으로 이어질 수 있어 FP보다 더 큰 문제이므로, Sensitivity(민감도)가 Specificity(특이도)보다 중요하다.

**ROC Curve(중)**: X축은 False Positive Rate, Y축은 True Positive Rate를 나타낸다. 곡선이 좌상단에 가까울수록 성능이 우수하며, 대각선(AUC=0.5)은 무작위 분류기 수준이다. **AUC > 0.7** 이면 실용적 수준의 분류 성능으로 평가한다.

**성능 지표 막대(우)**: 네 가지 지표를 종합 비교한다.
- **Balanced Accuracy**: 클래스 불균형 상황에서 정확도를 보정한 지표 (0.5 = 무작위)
- **ROC-AUC**: 임계값에 무관한 전반적 분류 능력
- **MCC**: 클래스 불균형에 강건한 상관계수 기반 지표 (0 = 무작위, 1 = 완벽)
- **F1**: 정밀도와 재현율의 조화 평균

모든 지표가 0.5(점선) 이상일 때 의미 있는 예측력을 가진다고 판단한다.

## 5. 모델 저장

In [ ]:
save_dict = {
    'task'             : 'skin_sensitization_classification',
    'dataset'          : 'NTP ICE Skin Sensitization (LLNA in vivo)',
    'best_descriptor'  : 'MACCS only',
    'best_model_name'  : 'SVM',
    'model'            : final_clf,
    'optimal_threshold': best_thr,
    'test_metrics': {
        'balanced_accuracy': best_bal,
        'roc_auc'          : roc_auc,
        'mcc'              : mcc,
        'f1'               : f1,
    },
}
joblib.dump(save_dict, 'skin_sensitization_model.joblib')
print('skin_sensitization_model.joblib 저장 완료')
print(f'  Balanced Accuracy: {best_bal:.4f}')
print(f'  ROC-AUC          : {roc_auc:.4f}')

## 6. 새 분자 예측

In [ ]:
loaded       = joblib.load('skin_sensitization_model.joblib')
model_loaded = loaded['model']
opt_thr      = loaded['optimal_threshold']

def predict_sensitization(smiles_list):
    remover_ = SaltRemover()
    results  = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            results.append({'SMILES': smi, 'Prediction': 'Invalid SMILES'})
            continue
        mol   = remover_.StripMol(mol)
        X_new = np.array(MACCSkeys.GenMACCSKeys(mol)).reshape(1, -1)
        proba = model_loaded.predict_proba(X_new)[0, 1]
        pred  = int(proba >= opt_thr)
        label = '감작성 (Active)' if pred == 1 else '비감작성 (Inactive)'
        results.append({'SMILES': smi, 'Prob_Active': round(proba, 4),
                        'Prediction': pred, 'Label': label})
    return pd.DataFrame(results)

# 테스트: DNCB (강한 감작성), 헥산 (비감작성), 에틸 데카노에이트 (비감작성)
test_smiles = [
    'Clc1ccc([N+](=O)[O-])cc1[N+](=O)[O-]',   # DNCB (강한 감작성)
    'CCCCCC',                                    # 헥산 (비감작성)
    'O=C(CCCCCCCC)OCC',                         # 에틸 데카노에이트 (비감작성)
]
print(predict_sensitization(test_smiles).to_string(index=False))

### 📊 예측 결과 분석

세 가지 대표 화합물로 모델의 실용적 예측 성능을 검증한다.

- **DNCB (2,4-Dinitrochlorobenzene)**: 실험적으로 확인된 강한 피부 감작성 물질(양성 대조군). 모델이 높은 Prob_Active(≥0.5)와 함께 감작성(Active)으로 예측해야 한다.
- **헥산(Hexane)**: 지방족 탄화수소로, 반응성 작용기가 없어 감작성이 없는 물질. 낮은 Prob_Active와 함께 비감작성(Inactive)으로 예측되어야 한다.
- **에틸 데카노에이트**: 지방산 에스테르 계열로 감작 가능성이 낮은 물질. 비감작성으로 예측되어야 한다.

세 물질 모두 예상 결과와 일치하면 모델이 감작성 물질의 구조적 특징을 올바르게 학습했음을 시사한다.

---
## 📋 실험 요약

| 항목 | 내용 |
|---|---|
| 데이터 | NTP ICE Skin Sensitization (LLNA in vivo) |
| 라벨 기준 | Active=감작성(1) / Inactive=비감작성(0) |
| 고유 화합물 | ~1,154개 (중복 제거 후) |
| 디스크립터 | MACCS only (167 bit) |
| 모델 | SVM (RBF, class_weight=balanced) |
| 검증 | 5-Fold Stratified CV |
| 평가 지표 | Balanced Accuracy, ROC-AUC, MCC, F1 |